In [ ]:
# rende config.py (in notebooks/) importabile anche dai notebook in sottocartelle
import sys, os
_cfg = os.getcwd()
while _cfg != os.path.dirname(_cfg):
    if os.path.exists(os.path.join(_cfg, 'config.py')):
        sys.path.insert(0, _cfg)
        break
    _cfg = os.path.dirname(_cfg)
from config import CONFESS_DATA, BC_DATA, ERA5_ROOT, POST_DATA, WORK_DIR, FIG_DIR, FIG_DIR_2025
import xarray as xr
import matplotlib.pyplot as plt
plt.switch_backend("Agg")  # backend headless: nel batch evita la ritenzione delle figure e riduce la RAM
import concurrent.futures
from concurrent.futures import ProcessPoolExecutor
import logging
import albedo_functions as af


In [ ]:
# Configurazione logging
log_file = "04-Fig4_BIAS_plot_seasons.log"
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s",
                    handlers=[logging.FileHandler(log_file)])


In [ ]:
exp_ctrl = 'a1ua'
exp_sens = 'a52o'
var = 'tas'
seasons = ['DJF', 'MAM', 'JJA', 'SON']
SAVE_PATH = f'{WORK_DIR}/'


In [ ]:
# La funzione di plotting sta in un modulo importabile (bias_seasons_lib.py), non
# qui: i processi 'spawn' usati sotto non vedono le funzioni definite nel notebook.
sys.path.insert(0, os.getcwd())   # rende importabile bias_seasons_lib
from bias_seasons_lib import plot_bias_season


In [ ]:
%%time
import multiprocessing as mp
# Un processo FRESCO per ogni figura (maxtasksperchild=1) con metodo 'spawn': il
# worker esce dopo ogni task e il SO recupera tutta la memoria (map_plot perde
# memoria a ogni figura). Uso multiprocessing.Pool perche' supporta maxtasksperchild
# gia' su Python 3.10 (ProcessPoolExecutor solo da 3.11).
from bias_seasons_lib import _plot_one
jobs = [(exp_ctrl, exp_sens, var, y1, y2, season, SAVE_PATH)
        for season in seasons for y1 in range(5) for y2 in range(y1 + 1, 5)]
with mp.get_context('spawn').Pool(processes=2, maxtasksperchild=1) as pool:
    for _i, msg in enumerate(pool.imap_unordered(_plot_one, jobs), 1):
        print(f"  {msg}   {_i}/{len(jobs)}", flush=True)
